Import libraries

In [1]:
from pathlib import Path
import re
import json

import joblib
import numpy as np
import pandas as pd

from IPython.display import display, clear_output
import ipywidgets as widgets

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json

from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator

DetectorFactory.seed = 42

print("Libraries imported successfully.")

I0000 00:00:1786881164.588494   21398 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786881164.606953   21398 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786881168.155915   21398 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786881181.127179   21398 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Libraries imported successfully.


Project paths

In [2]:
PROJECT_ROOT = Path.cwd()

# If notebook is inside notebooks/ folder
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "models"

LR_MODEL_PATH = MODELS_DIR / "logistic_regression_model.joblib"
TFIDF_PATH = MODELS_DIR / "tfidf_vectorizer.joblib"

LSTM_MODEL_PATH = MODELS_DIR / "lstm_model.keras"
LSTM_TOKENIZER_PATH = MODELS_DIR / "lstm_tokenizer.json"

MAX_LEN = 500

print("Project:", PROJECT_ROOT)
print("Models:", MODELS_DIR)

Project: /home/kinkini/Documents/NLP Dont delte/fake news detector/NLP_Group_05
Models: /home/kinkini/Documents/NLP Dont delte/fake news detector/NLP_Group_05/models


Check model files

In [3]:
required_files = [
    LR_MODEL_PATH,
    TFIDF_PATH,
    LSTM_MODEL_PATH,
    LSTM_TOKENIZER_PATH
]

for file_path in required_files:
    print(
        f"{file_path.name}:",
        "FOUND" if file_path.exists() else "NOT FOUND"
    )

logistic_regression_model.joblib: FOUND
tfidf_vectorizer.joblib: FOUND
lstm_model.keras: FOUND
lstm_tokenizer.json: FOUND


Load ML model

In [4]:
lr_model = joblib.load(LR_MODEL_PATH)

tfidf_vectorizer = joblib.load(TFIDF_PATH)

print("Logistic Regression model loaded.")
print("TF-IDF vectorizer loaded.")

Logistic Regression model loaded.
TF-IDF vectorizer loaded.


Load DL model

In [5]:
lstm_model = load_model(LSTM_MODEL_PATH)

with open(
    LSTM_TOKENIZER_PATH,
    "r",
    encoding="utf-8"
) as file:

    tokenizer_json = file.read()

lstm_tokenizer = tokenizer_from_json(tokenizer_json)

print("LSTM model loaded.")
print("LSTM tokenizer loaded.")

E0000 00:00:1786881187.217317   21398 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


LSTM model loaded.
LSTM tokenizer loaded.


Text preprocessing

In [6]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    tokens = text.split()

    tokens = [
        word
        for word in tokens
        if word not in stop_words
        and len(word) > 2
    ]

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)


print("Text preprocessing function ready.")

Text preprocessing function ready.


Logistic Regression prediction

In [7]:
def predict_logistic(news):

    cleaned_news = clean_text(news)

    news_tfidf = tfidf_vectorizer.transform(
        [cleaned_news]
    )

    prediction = lr_model.predict(
        news_tfidf
    )[0]

    probabilities = lr_model.predict_proba(
        news_tfidf
    )[0]

    fake_probability = probabilities[0]
    real_probability = probabilities[1]

    if prediction == 0:

        result = "Fake News"
        confidence = fake_probability

    else:

        result = "Real News"
        confidence = real_probability

    return {
        "model": "Logistic Regression",
        "prediction": result,
        "confidence": confidence,
        "fake_probability": fake_probability,
        "real_probability": real_probability
    }

LSTM prediction

In [8]:
def predict_lstm(news):

    cleaned_news = clean_text(news)

    sequence = lstm_tokenizer.texts_to_sequences(
        [cleaned_news]
    )

    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    prediction_probability = lstm_model.predict(
        padded_sequence,
        verbose=0
    )[0][0]

    real_probability = prediction_probability
    fake_probability = 1 - real_probability

    if real_probability >= 0.5:

        result = "Real News"
        confidence = real_probability

    else:

        result = "Fake News"
        confidence = fake_probability

    return {
        "model": "LSTM",
        "prediction": result,
        "confidence": confidence,
        "fake_probability": fake_probability,
        "real_probability": real_probability
    }

Test both models

In [9]:
sample_news = """
India’s KL Rahul and Devdutt Padikkal scored unbeaten half-centuries to help India reach 197 for one at tea on the first day of the opening Test against Sri Lanka in Galle. The pair built a strong partnership and gave India an excellent start after captain Shubman Gill chose to bat first. ([Reuters][2])"""

ml_result = predict_logistic(sample_news)
dl_result = predict_lstm(sample_news)

print("ML Result:")
print(ml_result)

print("\nDL Result:")
print(dl_result)

ML Result:
{'model': 'Logistic Regression', 'prediction': 'Real News', 'confidence': np.float64(0.5901207607161955), 'fake_probability': np.float64(0.40987923928380454), 'real_probability': np.float64(0.5901207607161955)}

DL Result:
{'model': 'LSTM', 'prediction': 'Real News', 'confidence': np.float32(0.9911268), 'fake_probability': np.float32(0.008873224), 'real_probability': np.float32(0.9911268)}


Create the UI

In [10]:
model_choice = widgets.Dropdown(
    options=[
        "Logistic Regression (ML)",
        "LSTM (DL)"
    ],
    value="Logistic Regression (ML)",
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)


news_input = widgets.Textarea(
    value="",
    placeholder="Enter or paste news text here...",
    description="News:",
    disabled=False,
    layout=widgets.Layout(
        width="1200px",
        height="200px"
    ),
    style={"description_width": "initial"}
)


predict_button = widgets.Button(
    description="Run Prediction",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(
        width="200px",
        height="40px"
    )
)


output = widgets.Output()

Prediction button

In [11]:
def run_prediction(button):

    with output:

        clear_output()

        news = news_input.value.strip()

        if not news:

            print("Please enter some news text.")

            return


        selected_model = model_choice.value


        # ============================================
        # LOGISTIC REGRESSION
        # ============================================

        if selected_model == "Logistic Regression (ML)":

            result = predict_logistic(news)

            print("=" * 50)
            print("LOGISTIC REGRESSION")
            print("=" * 50)

            print(
                "Prediction:",
                result["prediction"]
            )

            print(
                "Confidence:",
                f"{result['confidence'] * 100:.2f}%"
            )

            print(
                "Fake Probability:",
                f"{result['fake_probability'] * 100:.2f}%"
            )

            print(
                "Real Probability:",
                f"{result['real_probability'] * 100:.2f}%"
            )


        # ============================================
        # LSTM
        # ============================================

        elif selected_model == "LSTM (DL)":

            result = predict_lstm(news)

            print("=" * 50)
            print("LSTM")
            print("=" * 50)

            print(
                "Prediction:",
                result["prediction"]
            )

            print(
                "Confidence:",
                f"{result['confidence'] * 100:.2f}%"
            )

            print(
                "Fake Probability:",
                f"{result['fake_probability'] * 100:.2f}%"
            )

            print(
                "Real Probability:",
                f"{result['real_probability'] * 100:.2f}%"
            )


        # ============================================
        # BOTH MODELS
        # ============================================

        else:

            ml_result = predict_logistic(news)
            dl_result = predict_lstm(news)

            print("=" * 50)
            print("MODEL COMPARISON")
            print("=" * 50)

            print("\nLogistic Regression (ML)")
            print(
                "Prediction:",
                ml_result["prediction"]
            )

            print(
                "Confidence:",
                f"{ml_result['confidence'] * 100:.2f}%"
            )

            print("\nLSTM (DL)")
            print(
                "Prediction:",
                dl_result["prediction"]
            )

            print(
                "Confidence:",
                f"{dl_result['confidence'] * 100:.2f}%"
            )

            print("\n" + "=" * 50)

            if (
                ml_result["prediction"]
                == dl_result["prediction"]
            ):

                print(
                    "Both models agree:",
                    ml_result["prediction"]
                )

            else:

                print(
                    "Models disagree."
                )

Connect button and display UI

In [12]:
predict_button.on_click(run_prediction)

display(
    widgets.VBox([
        widgets.HTML(
            "<h1>📰 Fake News Detection</h1>"
        ),

        widgets.HTML(
            "<h3>Select Machine Learning / Deep Learning Model</h3>"
        ),

        model_choice,

        widgets.HTML(
            "<h3>Enter News Text</h3>"
        ),

        news_input,

        widgets.HTML(
            "<br>"
        ),

        predict_button,

        output
    ])
)